# Day 2: Large-Scale JSONL Data Processing for AI Datasets

## 1. Core Theory (Just-in-Time)

### The "Why"
In AI engineering, datasets are rarely small enough to fit comfortably in memory (RAM). When fine-tuning Large Language Models (LLMs), preparing Retrieval-Augmented Generation (RAG) knowledge bases, or evaluating model outputs, you will frequently encounter datasets that are gigabytes or terabytes in size. 

Loading a 50GB JSON file into memory with `json.load()` will crash most systems (Out of Memory - OOM error). The standard solution in the AI industry is **JSON Lines (JSONL)**. 

### The "How"
A JSONL file contains one valid JSON object per line. This structure allows us to:
1. **Stream the data:** Read, process, and write one line (or a small batch of lines) at a time, keeping memory usage constant regardless of file size.
2. **Parallelize processing:** Because each line is independent, it's easy to split a JSONL file across multiple CPU cores or machines.
3. **Handle corruption gracefully:** If a single line is malformed, we can catch the error, log it, and continue processing the rest of the file.

We will use modern Python features to handle this:
- **Generators (`yield`):** For lazy evaluation/streaming of data.
- **`pydantic`:** For robust, strictly-typed data validation on each line.


## 2. Code Implementation

This production-grade example demonstrates how to safely stream, validate, and process a large JSONL file. We use `pydantic` to ensure the data strictly conforms to our expected schema before any downstream processing (like embedding generation or fine-tuning).

In [ ]:
import json
import logging
from typing import Iterator, List, Optional
from pydantic import BaseModel, ValidationError, Field
from pathlib import Path

# Setup basic logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# 1. Define strict type models for our data
class DocumentRecord(BaseModel):
    """Schema for a single document in our AI dataset."""
    id: str = Field(..., description="Unique identifier for the document")
    text: str = Field(..., min_length=1, description="The content of the document")
    metadata: Optional[dict] = Field(default_factory=dict, description="Optional metadata for filtering (e.g., author, date)")

def generate_dummy_data(filepath: Path, num_records: int = 100) -> None:
    """Helper function to create a dummy JSONL file for testing."""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with filepath.open("w", encoding="utf-8") as f:
        for i in range(num_records):
            # Introduce a bad record to demonstrate error handling
            if i == 50:
                f.write('{"id": "bad_record", "text": "", "metadata": {"error": "empty text"}}\n')
                continue
            
            record = {
                "id": f"doc_{i}",
                "text": f"This is the content for document {i}. It contains valuable information.",
                "metadata": {"source": "synthetic", "batch": i // 10}
            }
            f.write(json.dumps(record) + "\n")
    logger.info(f"Created {num_records} records in {filepath}")

def process_jsonl_stream(filepath: Path, batch_size: int = 10) -> Iterator[List[DocumentRecord]]:
    """
    Reads a JSONL file line-by-line, validates data, and yields batches.
    
    Args:
        filepath: Path to the JSONL file.
        batch_size: Number of records to yield at once.
        
    Yields:
        A list of validated DocumentRecord objects.
    """
    if not filepath.exists():
        raise FileNotFoundError(f"Dataset not found at {filepath}")

    batch: List[DocumentRecord] = []
    line_number = 0

    # Streaming read: memory efficient
    with filepath.open("r", encoding="utf-8") as f:
        for line in f:
            line_number += 1
            line = line.strip()
            if not line:
                continue

            try:
                # 1. Parse JSON
                raw_dict = json.loads(line)
                # 2. Validate with Pydantic
                record = DocumentRecord(**raw_dict)
                batch.append(record)

            except json.JSONDecodeError as e:
                logger.error(f"Invalid JSON at line {line_number}: {e}")
            except ValidationError as e:
                logger.error(f"Validation failed at line {line_number} for ID {raw_dict.get('id', 'UNKNOWN')}: {e}")
            
            # 3. Yield batch when full
            if len(batch) >= batch_size:
                yield batch
                batch = [] # Reset batch

    # Yield remaining records
    if batch:
        yield batch

# Execution Block
if __name__ == "__main__":
    dataset_path = Path("./data/raw_dataset.jsonl")
    
    # Generate some dummy data first
    generate_dummy_data(dataset_path, num_records=105)
    
    # Process the stream
    total_processed = 0
    logger.info("Starting processing stream...")
    
    for batch in process_jsonl_stream(dataset_path, batch_size=20):
        # In a real scenario, you might send this batch to an embedding API (like OpenAI)
        # or load it into a Vector DB (like Qdrant)
        total_processed += len(batch)
        logger.info(f"Successfully processed a batch of {len(batch)}. Total so far: {total_processed}")
        
    logger.info("Processing complete.")


## 3. Common Pitfalls in Production

1. **Ignoring Encoding:** Always explicitly set `encoding="utf-8"` when opening files (`open(filepath, "r", encoding="utf-8")`). Systems default to different encodings (e.g., Windows defaults to cp1252), which will crash when encountering emojis or special characters common in web-scraped AI datasets.
2. **`json.loads()` vs `json.load()`:** 
   - `json.load(f)` reads the *entire* file object at once. Do not use this for large JSON files.
   - `json.loads(string)` parses a *string*. In JSONL processing, we use `json.loads(line)` inside a loop.
3. **Failing to validate inputs:** Upstream data pipelines frequently change. If you assume a `"metadata"` field is always a dictionary and suddenly it's `null`, your downstream embedding script will throw an unhandled `TypeError` three hours into a ten-hour job. Always use a validator like `pydantic`.
4. **Missing batch processing:** Sending data to APIs (like LLM endpoints or Vector DBs) one by one is incredibly slow due to network latency. Batching (as shown in the code) allows you to utilize batch-API endpoints for massive speedups.

## 4. Practical Lab / Homework

**Task:** Build a dataset filter for a fine-tuning pipeline.

1. **Setup:** Use the `generate_dummy_data` function above to create a dataset, but modify it to generate 1,000 records. Add a new field to the schema called `"quality_score"` (a float between 0.0 and 1.0).
2. **Filter & Transform:** Write a script that reads this JSONL file stream using the batching technique. 
3. **Validation:** Use `pydantic` to ensure `"quality_score"` is actually a float.
4. **Write:** If a record has a `"quality_score"` >= 0.8, write it out to a new file called `high_quality_dataset.jsonl`.
5. **Constraint:** You must process the file line-by-line and write line-by-line. The entire dataset must never be in memory at the same time.

Use the cell below to implement your solution.

In [ ]:
# Implement your Practical Lab solution here
from pathlib import Path
from typing import Iterator, List, Optional
from pydantic import BaseModel, Field
import json
import random

# 1. Define your new Pydantic schema with quality_score
class TrainingRecord(BaseModel):
    id: str
    text: str
    quality_score: float = Field(..., ge=0.0, le=1.0)
    metadata: dict = Field(default_factory=dict)

# 2. Write your implementation below
